# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_06 — GPU-Accelerated Backends Benchmark

### Research question

When should a quant research pipeline move beyond pandas/NumPy and use accelerated backends such as Numba, CuPy, JAX, Polars, or GPU DataFrames?

This notebook benchmarks a realistic finance feature workload across multiple computational backends.

It focuses on:

1. a reproducible synthetic return panel;
2. a vectorised NumPy baseline;
3. optional Numba JIT acceleration;
4. optional CuPy GPU array acceleration;
5. optional JAX JIT acceleration;
6. optional Polars DataFrame acceleration;
7. correctness checks across backends;
8. warm-up, transfer, and compilation overhead;
9. benchmark reporting and environment metadata;
10. practical rules for when acceleration is worth it.

The goal is not to claim that GPUs are always faster. The goal is to build a disciplined benchmark harness that helps decide whether a given finance workload should stay in NumPy or move to an accelerated backend.

## 1. Why this matters

In quant research, performance bottlenecks often appear in:

- large feature grids;
- rolling-window calculations;
- covariance estimation;
- Monte Carlo simulation;
- order-book event replay;
- cross-sectional ranking;
- repeated backtests;
- parameter sweeps;
- intraday data processing.

However, acceleration is not free.

A GPU or JIT backend may introduce:

- compilation overhead;
- host-to-device transfer cost;
- device-to-host transfer cost;
- memory constraints;
- dtype differences;
- debugging complexity;
- dependency and deployment issues.

A serious benchmark should therefore compare:

| Question | Why it matters |
|---|---|
| Does the accelerated backend match the reference result? | Speed is useless if the answer changes silently. |
| Is the workload large enough? | Small arrays may be slower on GPU due to overhead. |
| Is the workload vectorisable? | GPUs like large parallel arithmetic. |
| Is data already on device? | Transfer costs can dominate. |
| Does the backend compile? | JIT compilation time must be separated from execution time. |
| Is memory layout controlled? | Contiguous arrays and dtype choices matter. |

## 2. Benchmark philosophy

This notebook follows a simple rule:

> Start with a clear NumPy reference implementation, then compare alternative backends against it.

The benchmark has two workloads:

### Workload A — Array feature engine

Given a matrix of returns $R \in \mathbb R^{T \times N}$, compute:

1. rolling realised volatility;
2. cross-sectional z-scores;
3. momentum features;
4. covariance matrix;
5. a combined signal diagnostic.

This resembles high-volume feature computation in a multi-asset research pipeline.

### Workload B — Long-format grouped aggregation

Given a long table with columns:

```text
timestamp, symbol, return
```

compute group-level summary statistics.

This resembles vendor/raw-data table processing, where a DataFrame engine such as pandas, Polars, or cuDF may matter.

The exact fastest backend depends on hardware and environment. The notebook therefore records environment metadata alongside results.

## 3. Imports and environment capture

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import importlib
import json
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
def optional_import(module_name: str):
    """
    Try to import an optional module.

    Returns
    -------
    module:
        Imported module or None.

    error:
        None if import succeeded, otherwise the exception string.
    """
    try:
        module = importlib.import_module(module_name)
        return module, None
    except Exception as exc:
        return None, str(exc)


cupy, cupy_error = optional_import("cupy")
jax, jax_error = optional_import("jax")
jnp, jnp_error = optional_import("jax.numpy")
numba, numba_error = optional_import("numba")
polars, polars_error = optional_import("polars")

availability = pd.DataFrame([
    {"backend": "NumPy", "available": True, "note": "Baseline"},
    {"backend": "Numba", "available": numba is not None, "note": numba_error},
    {"backend": "CuPy", "available": cupy is not None, "note": cupy_error},
    {"backend": "JAX", "available": (jax is not None and jnp is not None), "note": jax_error or jnp_error},
    {"backend": "Polars", "available": polars is not None, "note": polars_error},
])

availability

In [ ]:
def capture_environment_metadata() -> dict:
    """
    Capture benchmark-relevant environment metadata.
    """
    metadata = {
        "python_version": sys.version,
        "platform": platform.platform(),
        "processor": platform.processor(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "numba_available": numba is not None,
        "cupy_available": cupy is not None,
        "jax_available": (jax is not None and jnp is not None),
        "polars_available": polars is not None,
    }

    if numba is not None:
        metadata["numba_version"] = getattr(numba, "__version__", "unknown")

    if cupy is not None:
        metadata["cupy_version"] = getattr(cupy, "__version__", "unknown")
        try:
            metadata["cupy_device_count"] = int(cupy.cuda.runtime.getDeviceCount())
            metadata["cupy_device_name"] = cupy.cuda.runtime.getDeviceProperties(0)["name"].decode()
        except Exception as exc:
            metadata["cupy_device_error"] = str(exc)

    if jax is not None:
        metadata["jax_version"] = getattr(jax, "__version__", "unknown")
        try:
            metadata["jax_devices"] = [str(device) for device in jax.devices()]
        except Exception as exc:
            metadata["jax_device_error"] = str(exc)

    if polars is not None:
        metadata["polars_version"] = getattr(polars, "__version__", "unknown")

    return metadata


environment_metadata = capture_environment_metadata()
environment_metadata

## 4. Synthetic multi-asset return panel

We generate synthetic returns using a factor model:

$$
r_t = Bf_t + \epsilon_t
$$
where:

- $B$ is the asset-factor loading matrix;
- $f_t$ contains common factor returns;
- $\epsilon_t$ is idiosyncratic noise.

This gives a realistic cross-sectional covariance structure while remaining reproducible.

The default array size is intentionally moderate so the notebook can run on ordinary laptops. Increase `num_steps` and `num_assets` if you want to stress test GPU backends.

In [ ]:
@dataclass(frozen=True)
class BenchmarkConfig:
    """
    Benchmark configuration.

    Increase num_steps and num_assets for more serious GPU benchmarking.
    """
    num_steps: int = 2_500
    num_assets: int = 500
    num_factors: int = 8
    rolling_window: int = 21
    momentum_lookback: int = 63
    dtype: str = "float64"
    seed: int = 42


config = BenchmarkConfig()
config

In [ ]:
def generate_synthetic_feature_panel(config: BenchmarkConfig) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate synthetic returns and prices for feature benchmarking.

    Returns
    -------
    returns:
        Array with shape (T, N).

    prices:
        Array with shape (T, N).
    """
    local_rng = np.random.default_rng(config.seed)

    T = config.num_steps
    N = config.num_assets
    K = config.num_factors
    dtype = np.float64 if config.dtype == "float64" else np.float32

    factor_loadings = local_rng.normal(loc=0.0, scale=0.6, size=(N, K))
    factor_vols = np.linspace(0.014, 0.004, K)

    factor_returns = local_rng.normal(
        loc=0.0,
        scale=factor_vols,
        size=(T, K)
    )

    idio_vols = local_rng.uniform(0.006, 0.025, size=N)

    idio_noise = local_rng.normal(
        loc=0.0,
        scale=idio_vols,
        size=(T, N)
    )

    returns = factor_returns @ factor_loadings.T + idio_noise
    returns = returns.astype(dtype)

    # Prevent extreme synthetic returns from creating absurd prices.
    returns = np.clip(returns, -0.25, 0.25)

    log_prices = np.log(100.0) + np.cumsum(np.log1p(returns), axis=0)
    prices = np.exp(log_prices).astype(dtype)

    return np.ascontiguousarray(returns), np.ascontiguousarray(prices)


returns_np, prices_np = generate_synthetic_feature_panel(config)

returns_np.shape, prices_np.shape, returns_np.dtype

## 5. Array workload definition

The array workload computes a set of common research features.

### 5.1 Rolling realised volatility

$$
\begin{aligned}
\sigma_{t,i}^{\text{realised}} &= \sqrt{ \frac{252}{w} \sum_{j=t-w+1}^{t} r_{j,i}^2 }
\end{aligned}
$$
where $w$ is the rolling window.

### 5.2 Cross-sectional z-score

$$
\begin{aligned}
z_{t,i} &= \frac{r_{t,i} - \bar r_t}{s_t}
\end{aligned}
$$
where $\bar r_t$ and $s_t$ are the cross-sectional mean and standard deviation across assets on date $t$.

### 5.3 Momentum

$$
\begin{aligned}
m_{t,i} &= \log(P_{t,i}) - \log(P_{t-L,i})
\end{aligned}
$$
where $L$ is the lookback period.

### 5.4 Covariance matrix

$$
\begin{aligned}
\hat\Sigma &= \frac{1}{T-1} X^\top X
\end{aligned}
$$
where $X$ is the demeaned return matrix.

In [ ]:
def numpy_feature_kernel(
    returns: np.ndarray,
    prices: np.ndarray,
    rolling_window: int,
    momentum_lookback: int
) -> dict:
    """
    NumPy baseline feature workload.

    Returns a compact set of scalar diagnostics so that all backends can
    be compared without transferring huge intermediate arrays.
    """
    returns = np.asarray(returns)
    prices = np.asarray(prices)

    squared = returns ** 2
    cumsum_squared = np.cumsum(squared, axis=0)
    rolling_sum = cumsum_squared[rolling_window:] - cumsum_squared[:-rolling_window]
    rolling_vol = np.sqrt(252.0 * rolling_sum / rolling_window)

    cross_mean = returns.mean(axis=1, keepdims=True)
    cross_std = returns.std(axis=1, keepdims=True) + 1e-12
    zscores = (returns - cross_mean) / cross_std

    log_prices = np.log(prices)
    momentum = log_prices[momentum_lookback:] - log_prices[:-momentum_lookback]

    demeaned = returns - returns.mean(axis=0, keepdims=True)
    covariance = (demeaned.T @ demeaned) / (returns.shape[0] - 1)

    # A compact synthetic signal diagnostic.
    aligned_rows = min(rolling_vol.shape[0], momentum.shape[0], zscores.shape[0])
    signal = (
        0.5 * np.tanh(zscores[-aligned_rows:])
        + 0.3 * np.tanh(momentum[-aligned_rows:])
        - 0.2 * np.tanh(rolling_vol[-aligned_rows:])
    )

    return {
        "avg_rolling_vol": float(np.mean(rolling_vol)),
        "mean_abs_zscore": float(np.mean(np.abs(zscores))),
        "avg_momentum": float(np.mean(momentum)),
        "covariance_trace": float(np.trace(covariance)),
        "signal_abs_mean": float(np.mean(np.abs(signal)))
    }


numpy_reference = numpy_feature_kernel(
    returns_np,
    prices_np,
    rolling_window=config.rolling_window,
    momentum_lookback=config.momentum_lookback
)

numpy_reference

## 6. Timing helper

Benchmarking compiled or GPU-backed code requires care.

The timing helper supports:

- warm-up runs;
- repeated measurements;
- median/min/max statistics;
- optional synchronisation after each run.

For GPU backends, synchronisation is important because operations can be asynchronous.

In [ ]:
def time_callable(
    name: str,
    func,
    repeats: int = 5,
    warmups: int = 1,
    sync_func=None
) -> dict:
    """
    Time a callable with warmups and repeated runs.
    """
    for _ in range(warmups):
        result = func()
        if sync_func is not None:
            sync_func(result)

    timings = []

    last_result = None

    for _ in range(repeats):
        start = time.perf_counter()
        last_result = func()
        if sync_func is not None:
            sync_func(last_result)
        end = time.perf_counter()
        timings.append(end - start)

    timings = np.asarray(timings)

    return {
        "backend": name,
        "min_seconds": float(timings.min()),
        "median_seconds": float(np.median(timings)),
        "max_seconds": float(timings.max()),
        "repeats": repeats
    }


benchmark_rows = []
correctness_rows = []

In [ ]:
numpy_timing = time_callable(
    name="NumPy",
    func=lambda: numpy_feature_kernel(
        returns_np,
        prices_np,
        config.rolling_window,
        config.momentum_lookback
    ),
    repeats=5,
    warmups=1
)

benchmark_rows.append(numpy_timing)
numpy_timing

## 7. Optional Numba backend

Numba can compile numerical Python loops into machine code.

This is especially useful when:

- the code is loop-heavy;
- vectorisation would be awkward;
- the workload stays on CPU;
- you want a lower engineering burden than C++.

The first Numba call includes compilation overhead, so we separate warm-up from timed runs.

In [ ]:
if numba is not None:
    from numba import njit

    @njit
    def numba_feature_kernel_core(returns, prices, rolling_window, momentum_lookback):
        T, N = returns.shape

        # Rolling volatility average.
        rolling_count = T - rolling_window
        rolling_vol_sum = 0.0
        rolling_vol_n = 0

        for t in range(rolling_window, T):
            for i in range(N):
                s = 0.0
                for k in range(t - rolling_window, t):
                    s += returns[k, i] * returns[k, i]
                rolling_vol_sum += np.sqrt(252.0 * s / rolling_window)
                rolling_vol_n += 1

        avg_rolling_vol = rolling_vol_sum / rolling_vol_n

        # Cross-sectional z-score average absolute value.
        z_abs_sum = 0.0
        z_count = 0

        for t in range(T):
            mean = 0.0
            for i in range(N):
                mean += returns[t, i]
            mean /= N

            var = 0.0
            for i in range(N):
                diff = returns[t, i] - mean
                var += diff * diff
            std = np.sqrt(var / N) + 1e-12

            for i in range(N):
                z_abs_sum += abs((returns[t, i] - mean) / std)
                z_count += 1

        mean_abs_zscore = z_abs_sum / z_count

        # Momentum average.
        momentum_sum = 0.0
        momentum_count = 0

        for t in range(momentum_lookback, T):
            for i in range(N):
                momentum_sum += np.log(prices[t, i]) - np.log(prices[t - momentum_lookback, i])
                momentum_count += 1

        avg_momentum = momentum_sum / momentum_count

        # Covariance trace = sum of sample variances.
        covariance_trace = 0.0

        for i in range(N):
            mean = 0.0
            for t in range(T):
                mean += returns[t, i]
            mean /= T

            var = 0.0
            for t in range(T):
                diff = returns[t, i] - mean
                var += diff * diff

            covariance_trace += var / (T - 1)

        # Compact signal proxy, kept simple for loop implementation.
        signal_abs_mean = 0.5 * mean_abs_zscore + 0.3 * abs(avg_momentum) + 0.2 * avg_rolling_vol

        return avg_rolling_vol, mean_abs_zscore, avg_momentum, covariance_trace, signal_abs_mean


    def numba_feature_kernel(returns, prices, rolling_window, momentum_lookback):
        values = numba_feature_kernel_core(
            returns,
            prices,
            rolling_window,
            momentum_lookback
        )

        return {
            "avg_rolling_vol": float(values[0]),
            "mean_abs_zscore": float(values[1]),
            "avg_momentum": float(values[2]),
            "covariance_trace": float(values[3]),
            "signal_abs_mean": float(values[4])
        }

    print("Numba kernel defined.")
else:
    print("Numba unavailable; skipping Numba backend.")

In [ ]:
if numba is not None:
    # Use a smaller array for the loop-style Numba benchmark to keep runtime reasonable.
    numba_returns = returns_np[:800, :200]
    numba_prices = prices_np[:800, :200]

    numba_result = numba_feature_kernel(
        numba_returns,
        numba_prices,
        config.rolling_window,
        config.momentum_lookback
    )

    numba_reference = numpy_feature_kernel(
        numba_returns,
        numba_prices,
        config.rolling_window,
        config.momentum_lookback
    )

    for key in ["avg_rolling_vol", "mean_abs_zscore", "avg_momentum", "covariance_trace"]:
        correctness_rows.append({
            "backend": "Numba",
            "metric": key,
            "reference": numba_reference[key],
            "backend_value": numba_result[key],
            "absolute_error": abs(numba_reference[key] - numba_result[key])
        })

    numba_timing = time_callable(
        name="Numba",
        func=lambda: numba_feature_kernel(
            numba_returns,
            numba_prices,
            config.rolling_window,
            config.momentum_lookback
        ),
        repeats=5,
        warmups=1
    )

    benchmark_rows.append(numba_timing)
    display(numba_timing)
else:
    print("Numba benchmark skipped.")

## 8. Optional CuPy GPU backend

CuPy provides a NumPy-like API for GPU arrays.

A key benchmarking distinction is:

1. **transfer-inclusive timing** — includes moving data from CPU to GPU and back;
2. **device-resident timing** — assumes data is already on GPU.

For research pipelines, this distinction is critical.

If the workload is small and data begins on CPU, transfer overhead may dominate.

If a large simulation or feature grid stays on GPU for many operations, GPU acceleration can be very valuable.

In [ ]:
if cupy is not None:
    cp = cupy

    def cupy_synchronise(_=None):
        cp.cuda.Stream.null.synchronize()

    def cupy_feature_kernel_device(returns_cp, prices_cp, rolling_window, momentum_lookback):
        squared = returns_cp ** 2
        cumsum_squared = cp.cumsum(squared, axis=0)
        rolling_sum = cumsum_squared[rolling_window:] - cumsum_squared[:-rolling_window]
        rolling_vol = cp.sqrt(252.0 * rolling_sum / rolling_window)

        cross_mean = cp.mean(returns_cp, axis=1, keepdims=True)
        cross_std = cp.std(returns_cp, axis=1, keepdims=True) + 1e-12
        zscores = (returns_cp - cross_mean) / cross_std

        log_prices = cp.log(prices_cp)
        momentum = log_prices[momentum_lookback:] - log_prices[:-momentum_lookback]

        demeaned = returns_cp - cp.mean(returns_cp, axis=0, keepdims=True)
        covariance = (demeaned.T @ demeaned) / (returns_cp.shape[0] - 1)

        aligned_rows = min(rolling_vol.shape[0], momentum.shape[0], zscores.shape[0])
        signal = (
            0.5 * cp.tanh(zscores[-aligned_rows:])
            + 0.3 * cp.tanh(momentum[-aligned_rows:])
            - 0.2 * cp.tanh(rolling_vol[-aligned_rows:])
        )

        output = cp.asarray([
            cp.mean(rolling_vol),
            cp.mean(cp.abs(zscores)),
            cp.mean(momentum),
            cp.trace(covariance),
            cp.mean(cp.abs(signal))
        ])

        return output

    def cupy_feature_kernel_transfer_inclusive(returns, prices, rolling_window, momentum_lookback):
        returns_cp = cp.asarray(returns)
        prices_cp = cp.asarray(prices)
        output = cupy_feature_kernel_device(
            returns_cp,
            prices_cp,
            rolling_window,
            momentum_lookback
        )
        cp.cuda.Stream.null.synchronize()
        values = cp.asnumpy(output)

        return {
            "avg_rolling_vol": float(values[0]),
            "mean_abs_zscore": float(values[1]),
            "avg_momentum": float(values[2]),
            "covariance_trace": float(values[3]),
            "signal_abs_mean": float(values[4])
        }

    returns_cp = cp.asarray(returns_np)
    prices_cp = cp.asarray(prices_np)
    cupy_synchronise()

    def cupy_feature_kernel_device_resident():
        output = cupy_feature_kernel_device(
            returns_cp,
            prices_cp,
            config.rolling_window,
            config.momentum_lookback
        )
        cp.cuda.Stream.null.synchronize()
        values = cp.asnumpy(output)

        return {
            "avg_rolling_vol": float(values[0]),
            "mean_abs_zscore": float(values[1]),
            "avg_momentum": float(values[2]),
            "covariance_trace": float(values[3]),
            "signal_abs_mean": float(values[4])
        }

    print("CuPy backend ready.")
else:
    print("CuPy unavailable; skipping CuPy backend.")

In [ ]:
if cupy is not None:
    cupy_result = cupy_feature_kernel_transfer_inclusive(
        returns_np,
        prices_np,
        config.rolling_window,
        config.momentum_lookback
    )

    for key in numpy_reference:
        correctness_rows.append({
            "backend": "CuPy",
            "metric": key,
            "reference": numpy_reference[key],
            "backend_value": cupy_result[key],
            "absolute_error": abs(numpy_reference[key] - cupy_result[key])
        })

    cupy_timing_transfer = time_callable(
        name="CuPy_transfer_inclusive",
        func=lambda: cupy_feature_kernel_transfer_inclusive(
            returns_np,
            prices_np,
            config.rolling_window,
            config.momentum_lookback
        ),
        repeats=5,
        warmups=1
    )

    cupy_timing_device = time_callable(
        name="CuPy_device_resident",
        func=cupy_feature_kernel_device_resident,
        repeats=5,
        warmups=1
    )

    benchmark_rows.extend([cupy_timing_transfer, cupy_timing_device])

    display(pd.DataFrame([cupy_timing_transfer, cupy_timing_device]))
else:
    print("CuPy benchmark skipped.")

## 9. Optional JAX backend

JAX provides NumPy-like array programming with JIT compilation and automatic differentiation.

The first `jax.jit` call includes compilation, so benchmark results should separate:

1. compilation + first execution;
2. repeated execution after compilation.

JAX is especially interesting for finance when the same computation must later be differentiated, for example:

- calibrating model parameters;
- differentiating Monte Carlo estimators;
- computing Greeks;
- learning stochastic dynamics;
- differentiable portfolio or execution simulation.

In [ ]:
if jax is not None and jnp is not None:
    import jax
    import jax.numpy as jnp

    @jax.jit
    def jax_feature_kernel_device(returns_jax, prices_jax, rolling_window: int, momentum_lookback: int):
        squared = returns_jax ** 2
        cumsum_squared = jnp.cumsum(squared, axis=0)
        rolling_sum = cumsum_squared[rolling_window:] - cumsum_squared[:-rolling_window]
        rolling_vol = jnp.sqrt(252.0 * rolling_sum / rolling_window)

        cross_mean = jnp.mean(returns_jax, axis=1, keepdims=True)
        cross_std = jnp.std(returns_jax, axis=1, keepdims=True) + 1e-12
        zscores = (returns_jax - cross_mean) / cross_std

        log_prices = jnp.log(prices_jax)
        momentum = log_prices[momentum_lookback:] - log_prices[:-momentum_lookback]

        demeaned = returns_jax - jnp.mean(returns_jax, axis=0, keepdims=True)
        covariance = (demeaned.T @ demeaned) / (returns_jax.shape[0] - 1)

        aligned_rows = min(rolling_vol.shape[0], momentum.shape[0], zscores.shape[0])
        signal = (
            0.5 * jnp.tanh(zscores[-aligned_rows:])
            + 0.3 * jnp.tanh(momentum[-aligned_rows:])
            - 0.2 * jnp.tanh(rolling_vol[-aligned_rows:])
        )

        return jnp.asarray([
            jnp.mean(rolling_vol),
            jnp.mean(jnp.abs(zscores)),
            jnp.mean(momentum),
            jnp.trace(covariance),
            jnp.mean(jnp.abs(signal))
        ])

    returns_jax = jnp.asarray(returns_np)
    prices_jax = jnp.asarray(prices_np)

    print("JAX backend ready.")
else:
    print("JAX unavailable; skipping JAX backend.")

In [ ]:
if jax is not None and jnp is not None:
    def jax_result_to_dict(output):
        output.block_until_ready()
        values = np.asarray(output)

        return {
            "avg_rolling_vol": float(values[0]),
            "mean_abs_zscore": float(values[1]),
            "avg_momentum": float(values[2]),
            "covariance_trace": float(values[3]),
            "signal_abs_mean": float(values[4])
        }

    # First call triggers compilation.
    start_compile = time.perf_counter()
    first_output = jax_feature_kernel_device(
        returns_jax,
        prices_jax,
        config.rolling_window,
        config.momentum_lookback
    )
    first_output.block_until_ready()
    compile_plus_first_seconds = time.perf_counter() - start_compile

    jax_result = jax_result_to_dict(first_output)

    for key in numpy_reference:
        correctness_rows.append({
            "backend": "JAX",
            "metric": key,
            "reference": numpy_reference[key],
            "backend_value": jax_result[key],
            "absolute_error": abs(numpy_reference[key] - jax_result[key])
        })

    def run_jax_compiled():
        output = jax_feature_kernel_device(
            returns_jax,
            prices_jax,
            config.rolling_window,
            config.momentum_lookback
        )
        output.block_until_ready()
        return output

    jax_timing = time_callable(
        name="JAX_jit_compiled",
        func=run_jax_compiled,
        repeats=5,
        warmups=1,
        sync_func=lambda out: out.block_until_ready()
    )

    jax_compile_row = {
        "backend": "JAX_compile_plus_first_run",
        "min_seconds": compile_plus_first_seconds,
        "median_seconds": compile_plus_first_seconds,
        "max_seconds": compile_plus_first_seconds,
        "repeats": 1
    }

    benchmark_rows.extend([jax_compile_row, jax_timing])

    display(pd.DataFrame([jax_compile_row, jax_timing]))
else:
    print("JAX benchmark skipped.")

## 10. Optional Polars DataFrame backend

The array workload is naturally suited to NumPy/CuPy/JAX.

But market data often begins in long-format tables.

For a DataFrame-style workload, we compare pandas with optional Polars.

The task is:

1. group by symbol;
2. compute mean return, volatility, skew-like third moment proxy, and observation count;
3. sort by volatility.

This is not a GPU benchmark, but it is relevant to Phase 1 data infrastructure because DataFrame engines can be a major bottleneck before the data ever reaches an array model.

In [ ]:
def make_long_return_frame(returns: np.ndarray) -> pd.DataFrame:
    """
    Convert return matrix into long-format DataFrame.
    """
    T, N = returns.shape
    dates = pd.date_range("2020-01-01", periods=T, freq="B", tz="UTC")
    symbols = [f"Asset_{i:04d}" for i in range(N)]

    return pd.DataFrame({
        "timestamp": np.repeat(dates.to_numpy(), N),
        "symbol": np.tile(symbols, T),
        "return": returns.reshape(-1)
    })


long_returns = make_long_return_frame(returns_np[:1_500, :300])
long_returns.head()

In [ ]:
def pandas_groupby_workload(df: pd.DataFrame) -> pd.DataFrame:
    """
    pandas groupby workload.
    """
    summary = (
        df
        .groupby("symbol")["return"]
        .agg(
            mean_return="mean",
            volatility="std",
            count="count"
        )
        .reset_index()
    )

    summary["mean_abs_return"] = (
        df.assign(abs_return=df["return"].abs())
        .groupby("symbol")["abs_return"]
        .mean()
        .to_numpy()
    )

    return summary.sort_values("volatility", ascending=False).reset_index(drop=True)


pandas_groupby_result = pandas_groupby_workload(long_returns)
pandas_groupby_result.head()

In [ ]:
pandas_groupby_timing = time_callable(
    name="pandas_groupby",
    func=lambda: pandas_groupby_workload(long_returns),
    repeats=5,
    warmups=1
)

benchmark_rows.append(pandas_groupby_timing)
pandas_groupby_timing

In [ ]:
if polars is not None:
    pl = polars

    long_returns_pl = pl.from_pandas(long_returns)

    def polars_groupby_workload(df_pl):
        return (
            df_pl
            .group_by("symbol")
            .agg([
                pl.col("return").mean().alias("mean_return"),
                pl.col("return").std().alias("volatility"),
                pl.col("return").count().alias("count"),
                pl.col("return").abs().mean().alias("mean_abs_return"),
            ])
            .sort("volatility", descending=True)
        )

    polars_result = polars_groupby_workload(long_returns_pl)

    polars_groupby_timing = time_callable(
        name="polars_groupby",
        func=lambda: polars_groupby_workload(long_returns_pl),
        repeats=5,
        warmups=1
    )

    benchmark_rows.append(polars_groupby_timing)

    display(polars_groupby_timing)
else:
    print("Polars unavailable; skipping Polars backend.")

## 11. Benchmark summary

We now combine all benchmark results into one table.

Interpretation rules:

1. compare like with like;
2. do not compare Numba small-array timing directly against full-array NumPy timing;
3. separate JAX compile time from post-compile execution time;
4. distinguish CuPy transfer-inclusive and device-resident timings;
5. repeat benchmarks on your own hardware before drawing conclusions.

In [ ]:
benchmark_df = pd.DataFrame(benchmark_rows)

# Compare speedup against NumPy array baseline where appropriate.
numpy_median = benchmark_df.loc[
    benchmark_df["backend"] == "NumPy",
    "median_seconds"
].iloc[0]

benchmark_df["speedup_vs_numpy_array_baseline"] = numpy_median / benchmark_df["median_seconds"]

benchmark_df.sort_values("median_seconds")

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = benchmark_df.sort_values("median_seconds", ascending=True)

plt.bar(plot_df["backend"], plot_df["median_seconds"])
plt.yscale("log")
plt.xticks(rotation=35, ha="right")
plt.title("Backend Runtime Comparison")
plt.xlabel("Backend")
plt.ylabel("Median seconds, log scale")
plt.show()

## 12. Correctness summary

Backends may differ slightly because of:

- dtype defaults;
- floating-point operation ordering;
- CPU/GPU numerical kernels;
- parallel reductions;
- compiler optimisations.

Small numerical differences are expected. Large differences are a warning sign.

In [ ]:
correctness_df = pd.DataFrame(correctness_rows)

if correctness_df.empty:
    print("No optional backends produced correctness diagnostics.")
else:
    display(correctness_df.sort_values("absolute_error", ascending=False).head(20))

## 13. Memory and transfer diagnostics

A GPU can be fast but still lose if most time is spent transferring arrays.

A rough memory estimate is:

$$
\text{memory bytes} = T \times N \times \text{bytes per element}
$$
For two matrices, returns and prices:

$$
\text{total bytes} \approx 2TNb
$$
where $b$ is 8 for `float64` and 4 for `float32`.

This matters because GPU speedups are strongest when data stays on device across many operations.

In [ ]:
def estimate_array_memory_mb(*arrays: np.ndarray) -> float:
    """
    Estimate total memory footprint of arrays in megabytes.
    """
    total_bytes = sum(arr.nbytes for arr in arrays)
    return total_bytes / (1024 ** 2)


memory_summary = pd.Series({
    "returns_mb": returns_np.nbytes / (1024 ** 2),
    "prices_mb": prices_np.nbytes / (1024 ** 2),
    "total_input_mb": estimate_array_memory_mb(returns_np, prices_np),
    "dtype": str(returns_np.dtype),
    "num_steps": returns_np.shape[0],
    "num_assets": returns_np.shape[1]
})

memory_summary

## 14. Backend selection guide

A practical rule of thumb:

| Workload | Usually start with | Consider moving to |
|---|---|---|
| Small notebook analysis | pandas / NumPy | Nothing unless slow |
| Large matrix arithmetic | NumPy | CuPy or JAX |
| Loop-heavy CPU code | Python reference | Numba |
| Repeated differentiable simulation | NumPy prototype | JAX |
| Large long-format tabular data | pandas | Polars or cuDF |
| Event-driven order-book replay | Python prototype | C++ / Numba / Rust |
| Huge Monte Carlo path simulation | NumPy | CuPy / JAX / C++ / CUDA |
| Production latency-sensitive code | Python prototype | C++ / Rust |

The key is to optimise the bottleneck, not the entire codebase.

## 15. Saving benchmark results

A useful benchmark should save:

1. timings;
2. correctness errors;
3. memory footprint;
4. environment metadata;
5. benchmark configuration.

Without environment metadata, benchmark results are hard to interpret.

In [ ]:
output_dir = Path("data/processed/backend_benchmark_v1")
output_dir.mkdir(parents=True, exist_ok=True)

benchmark_path = output_dir / "backend_benchmark_results.csv"
correctness_path = output_dir / "backend_correctness_checks.csv"
metadata_path = output_dir / "backend_environment_metadata.json"
config_path = output_dir / "backend_benchmark_config.json"

benchmark_df.to_csv(benchmark_path, index=False)

if not correctness_df.empty:
    correctness_df.to_csv(correctness_path, index=False)
else:
    pd.DataFrame(columns=["backend", "metric", "reference", "backend_value", "absolute_error"]).to_csv(
        correctness_path,
        index=False
    )

metadata_path.write_text(json.dumps(environment_metadata, indent=2, default=str))
config_path.write_text(json.dumps(asdict(config), indent=2))

benchmark_path, correctness_path, metadata_path, config_path

## 16. Limitations

This benchmark is deliberately controlled and simplified.

### 16.1 Synthetic data is not a full production workload

Real data pipelines include:

- I/O bottlenecks;
- decompression;
- joins;
- missing data handling;
- timestamp alignment;
- vendor reconciliation;
- corporate action processing;
- non-contiguous memory layouts.

This notebook isolates numerical feature computation.

### 16.2 Benchmark results are hardware-dependent

The fastest backend depends on:

- CPU model;
- GPU model;
- memory bandwidth;
- installed BLAS;
- CUDA/ROCm version;
- JAX/XLA version;
- array size;
- dtype;
- thermal throttling;
- current system load.

You should rerun the benchmark on the target machine.

### 16.3 GPU transfer overhead can dominate

If arrays start on CPU and only one small operation is performed on GPU, acceleration may be slower.

GPU backends are more compelling when:

1. the dataset is large;
2. many operations happen sequentially on device;
3. the workload has high arithmetic intensity;
4. transfer costs are amortised.

### 16.4 JIT compilation changes timing interpretation

JAX and Numba can be very fast after compilation, but first-run latency can be significant.

This matters for interactive research and one-off jobs.

### 16.5 Floating-point differences are expected

Parallel reductions may sum numbers in different orders.

This can cause small numerical differences across backends even when the algorithm is correct.

### 16.6 GPU memory is finite

Large universes and long histories can exceed GPU memory.

Out-of-core, streaming, chunking, and distributed approaches may be required.

## 17. Research frontier and current directions

Accelerated computational backends are increasingly important in quantitative finance, but the frontier is not simply “use a GPU”.

### 17.1 Accelerator-native research pipelines

A mature accelerated pipeline keeps data on the accelerator across many operations.

Instead of repeatedly transferring CPU arrays to GPU and back, the pipeline should keep:

- features;
- model states;
- simulation paths;
- gradients;
- calibration losses;

on device as long as possible.

A future notebook could build a fully device-resident Monte Carlo pricing pipeline.

### 17.2 Differentiable finance with JAX

JAX is especially interesting because it combines array programming, JIT compilation, vectorisation, and automatic differentiation.

This enables workflows where pricing, calibration, and risk sensitivities are connected.

A future notebook could implement a differentiable Monte Carlo option pricer and compute Greeks using automatic differentiation.

### 17.3 GPU DataFrame pipelines

Array acceleration is useful after data is matrix-shaped.

But many bottlenecks happen earlier in the table-processing stage.

GPU DataFrame tools such as cuDF and CPU-accelerated engines such as Polars target joins, groupbys, filters, and aggregations on large tabular datasets.

A future notebook could compare pandas, Polars, and cuDF on a synthetic tick-data cleaning workload.

### 17.4 Randomised and approximate algorithms

Sometimes the best speedup does not come from hardware.

Approximate numerical algorithms can reduce work:

- randomised PCA;
- sketching;
- approximate nearest neighbours;
- low-rank covariance updates;
- online algorithms;
- streaming estimators.

A future notebook could compare exact PCA against randomised PCA for a large covariance matrix.

### 17.5 Precision-aware finance

Many GPU workloads run faster in `float32`, but finance often uses `float64`.

The correct choice depends on the task.

Examples:

- `float32` may be acceptable for exploratory feature screening;
- `float64` is safer for risk aggregation, covariance inversion, and pricing;
- mixed precision may be dangerous if not validated.

A future notebook could quantify how VaR, covariance, and portfolio weights change under `float32` versus `float64`.

### 17.6 Distributed and out-of-core computation

Some datasets cannot fit into one machine's memory.

Future infrastructure may use:

- chunked Parquet;
- DuckDB;
- Dask;
- Ray;
- Spark;
- distributed JAX;
- GPU clusters.

A future notebook could compare chunked local processing against in-memory processing for a large synthetic OHLCV panel.

### 17.7 Benchmark governance

Benchmarks can become misleading if they are not versioned.

A serious research repository should record:

- code version;
- data shape;
- random seed;
- dtype;
- hardware;
- backend versions;
- compiler/JIT settings;
- warm-up policy.

This notebook saves benchmark metadata to make results auditable.

## 18. Suggested follow-up notebooks

This notebook naturally leads to:

1. **00_05_python_to_cpp_finance_kernel**  
   Comparing compiled C++ kernels with Python orchestration.

2. **02_13_multilevel_monte_carlo_pricing**  
   Applying accelerated backends to simulation-heavy pricing.

3. **03_14_information_coefficient_and_feature_decay**  
   Accelerating large feature-grid evaluation.

4. **04_05_pca_spectral_risk_analysis**  
   Benchmarking covariance and PCA computations.

5. **05_01_vectorized_backtest_engine**  
   Accelerating vectorised backtesting across many assets and parameters.

6. **06_06_microstructure_friction_cpp_core**  
   Moving event-driven microstructure logic to compiled systems code.

7. **07_01_multi_asset_cta_research_pipeline**  
   Deciding which parts of the end-to-end pipeline deserve acceleration.

## 19. Summary

This notebook built a benchmark harness for accelerated finance backends.

It compared:

- NumPy as the reference array backend;
- optional Numba for compiled CPU loops;
- optional CuPy for GPU array operations;
- optional JAX for JIT-compiled array computation;
- optional Polars for long-format DataFrame aggregation.

The key computational takeaway is:

> Acceleration should be measured, not assumed. Transfer cost, compilation latency, memory layout, dtype, and workload size determine whether a backend is actually faster.

The key financial takeaway is:

> Large-scale quant research needs fast computation, but speed must not come at the cost of silent numerical drift or irreproducible benchmark claims.

## 20. Further reading

### Accelerated Python and GPU arrays

- CuPy documentation.
- JAX documentation.
- Numba documentation.
- RAPIDS cuDF documentation.
- Polars documentation.

### Numerical benchmarking

- Warm-up versus timed execution.
- CPU/GPU synchronisation.
- Host-device transfer overhead.
- Floating-point reproducibility.
- Benchmark versioning and environment capture.

### Future extensions

- JAX automatic differentiation for Greeks.
- CuPy Monte Carlo path generation.
- RAPIDS cuDF for tick-data cleaning.
- Polars lazy query optimisation.
- Numba event-loop acceleration.
- C++/pybind11 kernels for production infrastructure.